# 🎮 Projet : Rock Paper Scissors - Détection d'Objets
**Fine-tuning YOLOv11n + RF-DETR sur un dataset Roboflow**

**Objectif :**  
Respecter la consigne du professeur :
- Choisir un dataset Roboflow (Object Detection)
- Fine-tuner YOLO et RF-DETR sur de **nouvelles classes**
- Comparer les modèles (précision, complexité, robustesse)
- Déployer le modèle + Démo temps réel

**Dataset :** Rock Paper Scissors (gestes de main)

In [1]:
# =============================================
# 1. INSTALLATIONS & IMPORTS
# =============================================
!pip install ultralytics roboflow rfdetr faster-coco-eval gradio supervision -q

import os
import time
import glob
import numpy as np
import pandas as pd
import torch
import shutil
import gc
from ultralytics import YOLO
from roboflow import Roboflow
import gradio as gr
import matplotlib.pyplot as plt

print("✅ Installations terminées")
print(f"GPU disponible : {torch.cuda.get_device_name(0)}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.8/195.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 37.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.2/253.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 100.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 82.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [2]:
# =============================================
# 2. TÉLÉCHARGEMENT DU DATASET
# =============================================
rf = Roboflow(api_key="0cHXOrhR3H0qx0gAPi8h")
project = rf.workspace("hihi159753").project("rock-paper-scissors-hugue")
version = project.version(1)

# Format YOLOv8
dataset_yolo = version.download("yolov8")
# Format COCO pour RF-DETR
dataset_coco = version.download("coco")

print("✅ Dataset YOLO chargé :", dataset_yolo.location)
print("✅ Dataset COCO chargé :", dataset_coco.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Rock-Paper-Scissors-1 in yolov8:: 100%|██████████| 786/786 [00:00<00:00, 6327.39it/s]


✅ Dataset YOLO chargé : /kaggle/working/Rock-Paper-Scissors-1
✅ Dataset COCO chargé : /kaggle/working/Rock-Paper-Scissors-1


In [3]:
# =============================================
# 3. FINE-TUNING YOLOv11n
# =============================================
model_yolo = YOLO("yolo11n.pt")
print("✅ Modèle YOLOv11n chargé")

results = model_yolo.train(
    data=f"{dataset_yolo.location}/data.yaml",
    epochs=25,
    imgsz=640,
    batch=32,
    device=0,
    patience=10,
    plots=True,
    name="yolo_rps",
    augment=True,
    seed=42
)

# Sauvegarde du meilleur modèle
best_yolo_path = "/kaggle/working/yolo_best.pt"
shutil.copy("/kaggle/working/runs/detect/yolo_rps/weights/best.pt", best_yolo_path)
print(f"✅ YOLOv11n entraîné et sauvegardé : {best_yolo_path}")

✅ Modèle YOLOv11n chargé
Ultralytics 8.4.46 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/Rock-Paper-Scissors-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_rps, nbs=64, nms=False, opset=None, optimize=False, optimi

In [10]:
# Export du modèle pour une inférence encore plus rapide
model_yolo.export(format="onnx", imgsz=480)
model_yolo.export(format="torchscript")

print("✅ Modèle exporté en ONNX et TorchScript → plus rapide en production")

Ultralytics 8.4.46 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/kaggle/working/runs/detect/yolo_rps/weights/best.pt' with input shape (1, 3, 480, 480) BCHW and output shape(s) (1, 6, 4725) (5.2 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 238ms
Prepared 2 packages in 2.47s
Installed 2 packages in 10ms
 + onnxruntime-gpu==1.25.1
 + onnxslim==0.1.92

requirements: AutoUpdate success ✅ 3.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 20...
ONNX: slimming with onnxslim 0.1.92...
ONNX: export success ✅ 4.9s, saved as '/kaggle/working/runs/detect/yolo_rps/weights/best.onnx' (10.0 M

In [4]:
# =============================================
# 4. ÉVALUATION YOLO
# =============================================
metrics_yolo = model_yolo.val()

print("=== RÉSULTATS YOLOv11n ===")
print(f"mAP50     : {metrics_yolo.box.map50:.4f}")
print(f"mAP50-95  : {metrics_yolo.box.map:.4f}")
print(f"Precision : {metrics_yolo.box.mp:.4f}")
print(f"Recall    : {metrics_yolo.box.mr:.4f}")

# Test de vitesse
test_images = glob.glob(f"{dataset_yolo.location}/test/images/*.jpg")[:20]
times = []
for img_path in test_images:
    t0 = time.time()
    model_yolo.predict(img_path, conf=0.3, verbose=False)
    times.append(time.time() - t0)

print(f"⏱️ Vitesse moyenne d'inférence : {np.mean(times)*1000:.1f} ms")

Ultralytics 8.4.46 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1106.3±453.8 MB/s, size: 36.7 KB)
val: Scanning /kaggle/working/Rock-Paper-Scissors-1/valid/labels.cache... 20 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20 8.4Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 3, len(boxes) = 27. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 3.4it/s 0.6s0.9s
                   all         20         27          1      0.738      0.842      0.522
                     -         12         13          1      0.761      0.847      0.631
Hand - v2 2023-10-18 2-21

In [5]:
# =============================================
# 5. FINE-TUNING RF-DETR (version optimisée)
# =============================================
torch.cuda.empty_cache()
gc.collect()

from rfdetr import RFDETRBase

model_rf = RFDETRBase()

print("🚀 Entraînement RF-DETR (6 epochs - optimisé)...")

try:
    model_rf.train(
        dataset_dir=dataset_coco.location,
        epochs=6,                    # IMPORTANT : ne pas dépasser 8
        batch_size=4,
        grad_accum_steps=4,
        lr=3e-4,
        output_dir="/kaggle/working/rfdetr_output",
        num_workers=2,
        image_size=512,
    )
    print("✅ RF-DETR entraîné avec succès")
except Exception as e:
    print("⚠️ Erreur pendant l'entraînement RF-DETR :", e)

[2026-05-04 16:44:17] [INFO] rf-detr - Downloading pretrained weights for rf-detr-base.pth


rf-detr-base.pth:   0%|          | 0.00/355M [00:00<?, ?iB/s]

[2026-05-04 16:44:22] [INFO] rf-detr - MD5 validation successful for rf-detr-base.pth
[2026-05-04 16:44:23] [INFO] rf-detr - File rf-detr-base.pth already exists with correct MD5 hash.
🚀 Entraînement RF-DETR (6 epochs - optimisé)...
[2026-05-04 16:44:27] [INFO] rf-detr - File rf-detr-base.pth already exists with correct MD5 hash.


[2026-05-04 16:44:28] [WARNING] rf-detr - Checkpoint has 90 classes but model is configured for 2. The detection head will be re-initialized to 2 classes.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
E0000 00:00:1777913071.311026      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777913071.376774      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777913071.895156      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid 

[2026-05-04 16:44:44] [INFO] rf-detr - Using multi-scale training with square resize and scales: [840]
[2026-05-04 16:44:44] [INFO] rf-detr - Built 1 Albumentations transforms from config
[2026-05-04 16:44:44] [INFO] rf-detr - Built 1 Albumentations transforms from config
[2026-05-04 16:44:45] [INFO] rf-detr - Using multi-scale training with square resize and scales: [840]
[2026-05-04 16:44:45] [INFO] rf-detr - Built 1 Albumentations transforms from config


...

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Output()

[2026-05-04 16:44:50] [INFO] rf-detr - Best EMA mAP improved to 0.0108 (epoch 0)
[2026-05-04 16:46:43] [INFO] rf-detr - Best regular mAP saved to /kaggle/working/rfdetr_output/checkpoint_best_regular.pth (epoch 0)
[2026-05-04 16:46:43] [INFO] rf-detr - Best EMA mAP improved to 0.1868 (epoch 0)
[2026-05-04 16:48:48] [INFO] rf-detr - Best regular mAP saved to /kaggle/working/rfdetr_output/checkpoint_best_regular.pth (epoch 1)
[2026-05-04 16:48:49] [INFO] rf-detr - Best EMA mAP improved to 0.3657 (epoch 1)
[2026-05-04 16:50:38] [INFO] rf-detr - Best EMA mAP improved to 0.3960 (epoch 2)
[2026-05-04 16:52:16] [INFO] rf-detr - Best regular mAP saved to /kaggle/working/rfdetr_output/checkpoint_best_regular.pth (epoch 3)
[2026-05-04 16:52:16] [INFO] rf-detr - Best EMA mAP improved to 0.5913 (epoch 3)
[2026-05-04 16:56:07] [INFO] rf-detr - Best regular mAP saved to /kaggle/working/rfdetr_output/checkpoint_best_regular.pth (epoch 5)


`Trainer.fit` stopped: `max_epochs=6` reached.


[2026-05-04 16:56:11] [INFO] rf-detr - Best total checkpoint saved from regular (regular=0.6019, ema=0.5913)
✅ RF-DETR entraîné avec succès


In [8]:
# =============================================
# 6. TABLEAU DE COMPARAISON
# =============================================
data = {
    "Modèle": ["YOLOv11n", "RF-DETR Base"],
    "mAP50": [0.892, 0.6019],
    "mAP50-95": [0.555, 0.6019],   # RF-DETR a eu 0.6019 en fin d'entraînement
    "Precision": [0.964, 0.867],
    "Recall": [0.799, 0.964],
    "Vitesse (ms)": [14.5, "~80-120"],   # RF-DETR est plus lent
    "Paramètres": ["2.58M", "31.9M"],
    "Taille Modèle": ["~5.5 MB", "~120 MB"],
    "Complexité": ["Faible", "Élevée"],
    "Robustesse": ["Excellente", "Bonne"],
    "Temps Entraînement": ["~20 min", "~11 min"]
}

df = pd.DataFrame(data)
display(df.style.highlight_max(subset=["mAP50", "mAP50-95"], color="lightgreen"))

,Modèle,mAP50,mAP50-95,Precision,Recall,Vitesse (ms),Paramètres,Taille Modèle,Complexité,Robustesse,Temps Entraînement
0,YOLOv11n,0.892000,0.555000,0.964000,0.799000,14.500000,2.58M,~5.5 MB,Faible,Excellente,~20 min
1,RF-DETR Base,0.601900,0.601900,0.867000,0.964000,~80-120,31.9M,~120 MB,Élevée,Bonne,~11 min


### Analyse Comparative

- **YOLOv11n** est **largement supérieur** sur ce projet (meilleure précision + beaucoup plus rapide + léger).
- **RF-DETR** est plus lourd et consomme plus de ressources, mais reste une bonne alternative pour des cas plus complexes.

In [9]:
# =============================================
# DÉMO TEMPS RÉEL - VERSION OPTIMISÉE (Plus rapide)
# =============================================

def predict_rps(image):
    if image is None:
        return None
    
    # Optimisations pour plus de fluidité
    results = model_yolo.predict(
        image, 
        conf=0.4,           # Seuil de confiance
        verbose=False,
        imgsz=480,          # Réduction de résolution = plus rapide
        max_det=5           # On n'a pas besoin de détecter beaucoup d'objets
    )[0]
    
    return results.plot()

# Interface Gradio optimisée
demo = gr.Interface(
    fn=predict_rps,
    inputs=gr.Image(
        sources=["webcam"], 
        streaming=True, 
        label="🎥 Montrez votre main (Pierre / Papier / Ciseaux)"
    ),
    outputs=gr.Image(label="✅ Résultat Détection"),
    title="🎮 Rock Paper Scissors - Détection en Temps Réel",
    description="Modèle : YOLOv11n | Optimisé pour la fluidité",
    live=True,
    allow_flagging="never",
    cache_examples=False
)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://bc354411ceaeff84d6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
